In [1]:
# Loaded variable 'df' from URI: /home/anna/tcc/Seleção e Transformação/Mortalidade/tabela_mortes_por_faixa_etária.csv
import pandas as pd
df = pd.read_csv(r'/home/anna/tcc/Seleção e Transformação/Mortalidade/tabela_mortes_por_faixa_etária.csv')

In [2]:
import pandas as pd

df = pd.read_csv(
    r'/home/anna/tcc/Seleção e Transformação/Mortalidade/tabela_mortes_por_faixa_etária.csv'
)

# Seleciona somente as colunas de total por ano
colunas_total = [
    coluna for coluna in df.columns
    if coluna.endswith('_TOTAL')
    and not coluna.endswith('_TOTAL_QUANTIDADE')
]

# Soma todas as regionais de saúde para cada ano
totais_por_ano = df[colunas_total].apply(pd.to_numeric, errors='coerce').sum()

# Transforma em DataFrame
resultado = totais_por_ano.reset_index()
resultado.columns = ['Ano', 'Total']

# Extrai somente o ano
resultado['Ano'] = resultado['Ano'].str.replace('_TOTAL', '').astype(int)

# Ordena por ano
resultado = resultado.sort_values('Ano').reset_index(drop=True)

print(resultado)

     Ano   Total
0   1979   316.0
1   1980   345.0
2   1981   374.0
3   1982   351.0
4   1983   399.0
5   1984   438.0
6   1985   432.0
7   1986   386.0
8   1987   446.0
9   1988   465.0
10  1989   534.0
11  1990   548.0
12  1991   565.0
13  1992   557.0
14  1993   585.0
15  1994   606.0
16  1995   636.0
17  1996   646.0
18  1997   704.0
19  1998   691.0
20  1999   680.0
21  2000   645.0
22  2001   724.0
23  2002   730.0
24  2003   842.0
25  2004   905.0
26  2005   875.0
27  2006   969.0
28  2007   976.0
29  2008  1072.0
30  2009  1071.0
31  2010  1147.0
32  2011  1283.0
33  2012  1299.0
34  2013  1291.0
35  2014  1397.0
36  2015  1427.0
37  2016  1527.0
38  2017  1623.0
39  2018  1581.0
40  2019  1667.0
41  2020  1754.0
42  2021  1637.0
43  2022  1715.0
44  2023  1750.0
45  2024  1811.0


In [3]:
REGIONAL_ALVO = "Divinópolis"
ANOS = list(range(1979, 2025))
FAIXAS_ETARIAS = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70 OU MAIS"]

# ── Filtra municípios da regional ────────────────────────────────────────────
df_divinopolis = df[df["REGIONAL DE SAÚDE"].str.strip() == REGIONAL_ALVO].copy()

# ── Tabela de municípios da regional ─────────────────────────────────────────
df_municipios = df_divinopolis[["MUNICÍPIO", "CÓDIGO IBGE"]].sort_values("MUNICÍPIO").reset_index(drop=True)
print(f"Municípios da regional de {REGIONAL_ALVO}: {len(df_municipios)}")
display(df_municipios)

# ── Total por ano (soma dos municípios da regional) ──────────────────────────
colunas_total_ano = [f"{ano}_TOTAL" for ano in ANOS]
df_soma_ano = (
    df_divinopolis[colunas_total_ano]
    .sum(skipna=True)
    .rename_axis("ANO")
    .reset_index(name="ÓBITOS")
)
df_soma_ano["ANO"] = ANOS
display(df_soma_ano)

# ── Total por faixa etária (soma de todos os anos e municípios) ─────────────
df_soma_faixa = pd.DataFrame({
    "FAIXA ETÁRIA": FAIXAS_ETARIAS,
    "ÓBITOS": [
        df_divinopolis[[f"{ano}_{faixa}" for ano in ANOS]].sum(skipna=True).sum()
        for faixa in FAIXAS_ETARIAS
    ],
})
display(df_soma_faixa)

print(f"\nTotal geral ({REGIONAL_ALVO}): {int(df_soma_ano['ÓBITOS'].sum())}")

Municípios da regional de Divinópolis: 55


,MUNICÍPIO,CÓDIGO IBGE
0,aguanil,3100807
1,araujos,3103900
2,arcos,3104205
3,bambui,3105103
4,bom despacho,3107406
5,bom sucesso,3108008
6,camacho,3110400
7,campo belo,3111200
8,cana verde,3111903
9,candeias,3112000


,ANO,ÓBITOS
0,1979,24.0
1,1980,19.0
2,1981,25.0
3,1982,18.0
4,1983,25.0
5,1984,23.0
6,1985,26.0
7,1986,25.0
8,1987,39.0
9,1988,29.0


,FAIXA ETÁRIA,ÓBITOS
0,0-9,0.0
1,10-19,1.0
2,20-29,17.0
3,30-39,166.0
4,40-49,468.0
5,50-59,623.0
6,60-69,589.0
7,70 OU MAIS,820.0



Total geral (Divinópolis): 2684


In [4]:
# ── Total por ano E por faixa etária (matriz ano x faixa) ────────────────────
df_soma_ano_faixa = pd.DataFrame(
    {
        faixa: [df_divinopolis[f"{ano}_{faixa}"].sum(skipna=True) for ano in ANOS]
        for faixa in FAIXAS_ETARIAS
    },
    index=ANOS,
)
df_soma_ano_faixa.index.name = "ANO"
df_soma_ano_faixa["TOTAL"] = df_soma_ano_faixa.sum(axis=1)
display(df_soma_ano_faixa)

,0-9,10-19,20-29,30-39,40-49,50-59,60-69,70 OU MAIS,TOTAL
ANO,,,,,,,,,
1979,0.0,0.0,0.0,1.0,8.0,2.0,8.0,5.0,24.0
1980,0.0,0.0,0.0,2.0,1.0,7.0,2.0,7.0,19.0
1981,0.0,0.0,1.0,0.0,9.0,7.0,4.0,4.0,25.0
1982,0.0,0.0,0.0,1.0,5.0,3.0,6.0,3.0,18.0
1983,0.0,0.0,2.0,2.0,3.0,6.0,7.0,5.0,25.0
1984,0.0,0.0,0.0,1.0,5.0,8.0,6.0,3.0,23.0
1985,0.0,0.0,0.0,2.0,7.0,7.0,4.0,6.0,26.0
1986,0.0,0.0,0.0,2.0,4.0,8.0,5.0,6.0,25.0
1987,0.0,0.0,0.0,4.0,4.0,7.0,10.0,14.0,39.0


População

In [5]:
# Loaded variable 'df' from URI: /home/anna/tcc/Seleção e Transformação/População/tabela_populacao_faixa_etaria.csv
import pandas as pd
df = pd.read_csv(r'/home/anna/tcc/Seleção e Transformação/População/tabela_populacao_faixa_etaria.csv')

In [6]:
REGIONAL_ALVO = "divinopolis"
ANOS_ALVO = [1970, 1980, 2022]

df_divinopolis = df[df["REGIONAL DE SAÚDE"].str.strip() == REGIONAL_ALVO].copy()

df_soma_populacao = pd.DataFrame({
    "ANO": ANOS_ALVO,
    "POPULAÇÃO": [df_divinopolis[f"{ano}_TOTAL"].sum(skipna=True) for ano in ANOS_ALVO],
})
display(df_soma_populacao)

,ANO,POPULAÇÃO
0,1970,336331.0
1,1980,420981.0
2,2022,666470.0


In [10]:
ANO_ALVO = 1980


soma_populacao = df[f"{ANO_ALVO}_TOTAL"].sum(skipna=True)

print(f"População total em {ANO_ALVO}: {soma_populacao:,.0f}")

População total em 1980: 7,339,191


In [7]:
# Loaded variable 'df' from URI: /home/anna/tcc/Seleção e Transformação/População/tabela_populacao_faixa_etaria.csv
import pandas as pd
df = pd.read_csv(r'/home/anna/tcc/Seleção e Transformação/População/populacao_interpolacao_extrapolacao.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/home/anna/tcc/Seleção e Transformação/População/populacao_interpolacao_extrapolacao.csv'

In [ ]:
REGIONAL_ALVO = "divinopolis"
ANOS_ALVO = [1970, 1980, 2022]

df_divinopolis = df[df["REGIONAL DE SAÚDE"].str.strip() == REGIONAL_ALVO].copy()

df_soma_populacao = pd.DataFrame({
    "ANO": ANOS_ALVO,
    "POPULAÇÃO": [df_divinopolis[f"{ano}_TOTAL"].sum(skipna=True) for ano in ANOS_ALVO],
})
display(df_soma_populacao)

ANOS_ALVO = [1970, 1980, 2022]

df_soma_populacao = pd.DataFrame({
    "ANO": ANOS_ALVO,
    "POPULAÇÃO": [
        pd.to_numeric(df[f"{ano}_TOTAL"], errors="coerce").sum(skipna=True)
        for ano in ANOS_ALVO
    ],
})

display(df_soma_populacao)


,ANO,POPULAÇÃO
0,1970,336331.0
1,1980,420981.0
2,2022,666470.0


,ANO,POPULAÇÃO
0,1970,5772606.0
1,1980,7339191.0
2,2022,10524280.0


Mortalidade taxa

In [ ]:
# Loaded variable 'df' from URI: /home/anna/tcc/Seleção e Transformação/População/mortalidade_taxa.csv
import pandas as pd
df = pd.read_csv(r'/home/anna/tcc/Seleção e Transformação/População/mortalidade_taxa.csv')